<a href="https://colab.research.google.com/github/sokrypton/7.571/blob/main/L6/sequence_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Sequence Analysis & Probabilities

**A hands-on tutorial using only Python and NumPy**

We're going to answer one deceptively simple question:

> **"If two protein sequences look similar, is that meaningful — or just coincidence?"**

By the end of this notebook you'll understand:
1. How to build a **null distribution** and compute a **p-value** from scratch
2. How **substitution matrices** (like BLOSUM) are derived from real evolutionary data
3. How **Smith-Waterman** local alignment works
4. How **E-values** correct for multiple comparisons when searching a whole genome

---
## Part 0 — Setup

We only need NumPy, matplotlib, and SciPy. Everything else we build ourselves.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
import time
from scipy.stats import binom, gumbel_r

np.random.seed(42)

# The 20 standard amino acids, ordered by Dayhoff biochemical groups:
#   Sulfur | Small | Acid/Amide | Basic | Hydrophobic | Aromatic
AA = 'CSTPAGNDEQHRKMILVFYW'
AA_INDEX = {a: i for i, a in enumerate(AA)}

# Dayhoff group definitions and boundary positions (for heatmap dividers)
DAYHOFF_GROUPS = {
    'C': 'Sulfur',
    'S': 'Small', 'T': 'Small', 'P': 'Small', 'A': 'Small', 'G': 'Small',
    'N': 'Acid/Amide', 'D': 'Acid/Amide', 'E': 'Acid/Amide', 'Q': 'Acid/Amide',
    'H': 'Basic', 'R': 'Basic', 'K': 'Basic',
    'M': 'Hydrophobic', 'I': 'Hydrophobic', 'L': 'Hydrophobic', 'V': 'Hydrophobic',
    'F': 'Aromatic', 'Y': 'Aromatic', 'W': 'Aromatic',
}
group_labels = [DAYHOFF_GROUPS[a] for a in AA]
GROUP_BOUNDS = [i for i in range(1, 20) if group_labels[i] != group_labels[i-1]]

print(f"Amino acid alphabet ({len(AA)} residues): {AA}")
print("Ordered by Dayhoff groups: Sulfur | Small | Acid/Amide | Basic | Hydrophobic | Aromatic")

---
## Part 1 — Are These Sequences Similar by Chance?

In [ ]:
# Two protein fragments — an oxidoreductase and a possible distant homolog
seq_a = "MRIIVALITGATGQIGRFAIAQLLAQCEVLGIDTATSEQVARQCVDAMKPGGTFYTCARVADRDQSFAAALQASLKAFGRID"
seq_b = "DRIASTNAVRETQIICSGYNAQSLLLSFSLLADSCCTEDKGKRCFNSKTRLGTAALGAFVALADSTQGHTLQAELGAFPKVC"

def percent_identity(s1, s2):
    """Fraction of positions where two sequences have the same residue."""
    assert len(s1) == len(s2), "Sequences must be same length"
    return sum(a == b for a, b in zip(s1, s2)) / len(s1)

obs_identity = percent_identity(seq_a, seq_b)

midline = ''.join('|' if a == b else ' ' for a, b in zip(seq_a, seq_b))
print(f"A: {seq_a}")
print(f"   {midline}")
print(f"B: {seq_b}")
print(f"\nLength: {len(seq_a)},  Identity: {obs_identity:.1%}")
print(f"\n~28% identity — is that real, or could you get that by chance?")

### Is this identity significant?

**Null hypothesis:** The two sequences are unrelated; any similarity is due to chance alone.

**Strategy:** Generate thousands of **random protein sequences** drawn from realistic amino acid frequencies (matching E. coli proteome composition) and measure their identity.

In [ ]:
# Amino acid frequencies from the E. coli K-12 proteome
AA_FREQ = np.array([
    0.033,                                          # C             Sulfur
    0.070, 0.058, 0.051, 0.087, 0.089,              # S T P A G     Small
    0.040, 0.047, 0.050, 0.038,                     # N D E Q       Acid/Amide
    0.034, 0.041, 0.081,                            # H R K         Basic
    0.015, 0.037, 0.085, 0.065,                     # M I L V       Hydrophobic
    0.040, 0.030, 0.010,                            # F Y W         Aromatic
])
AA_FREQ = AA_FREQ / AA_FREQ.sum()

def random_protein(length):
    """Generate a random protein with realistic amino acid frequencies."""
    return ''.join(np.random.choice(list(AA), size=length, p=AA_FREQ))

# Expected random identity = sum(p_i^2)
expected_random = np.sum(AA_FREQ ** 2)
print(f"Expected random identity (sum of p_i²): {expected_random:.1%}")
print(f"  (Uniform over 20 AAs would give {1/20:.1%})")
print()

# Simulate 100k random sequence pairs (vectorized)
n_trials = 100_000
L = len(seq_a)
seqs_a = np.random.choice(len(AA), size=(n_trials, L), p=AA_FREQ)
seqs_b = np.random.choice(len(AA), size=(n_trials, L), p=AA_FREQ)
null_identities = np.mean(seqs_a == seqs_b, axis=1)

print(f"Empirical null identity (L={L}): {null_identities.mean():.1%} ± {null_identities.std():.1%}")
print(f"Observed identity:               {obs_identity:.1%}")
print()
p_value = np.mean(null_identities >= obs_identity)
print(f"Empirical p-value: {p_value}")

In [ ]:
# Visualize the null distribution
p_match = expected_random
binom_samples = np.random.binomial(L, p_match, size=n_trials) / L

fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(0, 0.20, 50)
ax.hist(null_identities, bins=bins, density=True, alpha=0.7, color='steelblue',
        edgecolor='white', label='Empirical (100k random pairs)')
ax.hist(binom_samples, bins=bins, density=True, alpha=0.4, color='black',
        edgecolor='white', label=f'Binomial(L={L}, p={p_match:.3f})')

ax.set_xlabel('Percent Identity', fontsize=13)
ax.set_ylabel('Density', fontsize=13)
ax.set_title(f'Null Distribution of Sequence Identity (L = {L})', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### How does sequence length affect what counts as "significant"?

In [ ]:
lengths = [20, 30, 50, 80, 100, 150, 200, 300, 500]
results = []

print(f"p_match = {p_match:.4f}")
print()
print(f"{'Length':>6}  {'E[id]':>7}  {'Std[id]':>8}  {'p<0.05 threshold':>16}  {'p<0.01 threshold':>16}")
print("-" * 65)

for Lv in lengths:
    t05 = binom.ppf(0.95, Lv, p_match) / Lv
    t01 = binom.ppf(0.99, Lv, p_match) / Lv
    std = np.sqrt(p_match * (1 - p_match) / Lv)
    results.append({'L': Lv, 'mean': p_match, 'std': std, 't05': t05, 't01': t01})
    print(f"{Lv:>6}  {p_match:>7.1%}  {std:>8.1%}  {t05:>16.1%}  {t01:>16.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
Ls = [r['L'] for r in results]
ax.plot(Ls, [r['t05'] for r in results], 'o-', color='orange', linewidth=2, markersize=8, label='p < 0.05')
ax.plot(Ls, [r['t01'] for r in results], 's-', color='red', linewidth=2, markersize=8, label='p < 0.01')
ax.axhline(p_match, color='gray', linestyle='--', linewidth=1, label=f'Random expectation ({p_match:.1%})')
ax.axhline(obs_identity, color='blue', linestyle=':', linewidth=1.5, label=f'Our observed: {obs_identity:.1%}')
ax.set_xlabel('Sequence Length', fontsize=13)
ax.set_ylabel('Percent Identity', fontsize=13)
ax.set_title('Identity Needed for Significance\nvs. Sequence Length', fontsize=14)
ax.legend(fontsize=9, loc='upper right')
ax.set_ylim(0, 0.35)
ax.grid(True, alpha=0.3)

ax = axes[1]
colors = plt.cm.viridis(np.linspace(0, 0.9, len(lengths)))
for Lv, color in zip(lengths, colors):
    k = np.arange(0, Lv + 1)
    ax.plot(k / Lv, binom.pmf(k, Lv, p_match) * Lv, color=color, linewidth=1.5, label=f'L={Lv}')
ax.axvline(obs_identity, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Percent Identity', fontsize=13)
ax.set_ylabel('Density', fontsize=13)
ax.set_title('Binomial Null Distributions\nNarrow with Increasing Length', fontsize=14)
ax.legend(fontsize=9, ncol=2)
ax.set_xlim(0, 0.30)

plt.tight_layout()
plt.show()

print(f"\nKey insight: at length {len(seq_a)}, our {obs_identity:.1%} identity is highly significant.")
print(f"But that same {obs_identity:.1%} at length 20 would barely clear the noise.")

### What did we learn?

The **p-value** is the probability of seeing similarity *this extreme or more* under the null hypothesis.

Three key insights:

1. **Each position is a coin flip.** Whether two random residues match follows a Bernoulli distribution with $p = \sum p_i^2 \approx 5.9\%$
2. **Longer sequences make the test more powerful.** The binomial distribution narrows, so smaller deviations become significant
3. **Amino acid composition matters.** Biased frequencies inflate random identity

But identity treats all mismatches equally — a conservative substitution (I → L) is scored the same as a radical one (I → D). We need a smarter scoring system.

---
## Part 2 — Building a Substitution Matrix from Evolutionary Data

### The idea behind BLOSUM

The BLOSUM (BLOcks SUbstitution Matrix) captures how often amino acids substitute for each other in evolution.

Given a **Multiple Sequence Alignment (MSA)**, we:
1. Count how often each pair (a, b) appears in the same column
2. Compare to how often we'd expect that pair by chance
3. Compute the **log-odds score**:

$$s_{ab} = \log_2 \frac{q_{ab}}{p_a \cdot p_b}$$

Let's build this from a real MSA!

### Downloading a real MSA(s) from the AlphaFold database

In [ ]:
def parse_a3m(text):
    """Parse A3M format: uppercase + dashes = aligned columns, lowercase = inserts (removed)."""
    sequences, current = [], []
    for line in text.strip().split('\n'):
        if line.startswith('>'):
            if current:
                sequences.append(''.join(current))
            current = []
        else:
            current.append(line.strip())
    if current:
        sequences.append(''.join(current))
    return [''.join(c for c in seq if c.isupper() or c == '-') for seq in sequences]

In [ ]:
# Download MSAs for multiple E. coli ribosomal proteins
# Using several to accumulate more diverse pair counts (closer to how BLOSUM was built)
RIBOSOMAL_PROTEINS = {
    'P0AG67': 'S1',  'P0A7V0': 'S2',  'P0A7V3': 'S3',  'P0A7V8': 'S4',
    'P0A7W1': 'S5',  'P0A7W7': 'S7',  'P0A7X3': 'S9',  'P0A7R5': 'S10',
    'P0A7R1': 'L1',  'P60723': 'L2',  'P60438': 'L3',  'P60723': 'L4',
    'P62399': 'L5',  'P0AG55': 'L6',  'P0A7J7': 'L10', 'P0AA10': 'L13',
}

all_msas = {}
for uniprot_id, name in RIBOSOMAL_PROTEINS.items():
    url = f"https://alphafold.ebi.ac.uk/files/msa/AF-{uniprot_id}-F1-msa_v6.a3m"
    try:
        response = urllib.request.urlopen(url)
        text = response.read().decode('utf-8')
        msa = parse_a3m(text)
        all_msas[name] = msa
        print(f"  {name} ({uniprot_id}): {len(msa)} sequences × {len(msa[0])} columns")
    except Exception as e:
        print(f"  {name} ({uniprot_id}): FAILED — {e}")

print(f"\nDownloaded {len(all_msas)} MSAs")

In [ ]:

msa = parse_a3m(a3m_text)
print(f"Parsed {len(msa)} sequences, alignment length {len(msa[0])}")
for i, seq in enumerate(msa[:5]):
    print(f"  Seq {i}: {seq[:60]}...")

### Now let's count substitution pairs and compute the matrix

In [ ]:
def build_substitution_matrix(msas, max_seqs=10000, max_identity=0.62):
    """
    Build a log-odds substitution matrix from multiple MSAs.

    Filters to sequence pairs within max_identity of the query (first sequence),
    mimicking BLOSUM's clustering at 62% identity.
    """
    pair_counts = np.zeros((20, 20))
    single_counts = np.zeros(20)
    total_columns = 0

    for name, msa in msas.items():
        seqs = msa[:max_seqs]
        n_seqs, aln_len = len(seqs), len(seqs[0])

        # Convert to integer array
        msa_int = np.full((n_seqs, aln_len), -1, dtype=np.int8)
        for i, seq in enumerate(seqs):
            for j, c in enumerate(seq):
                if c in AA_INDEX:
                    msa_int[i, j] = AA_INDEX[c]

        # Filter: keep only sequences within max_identity of the query (row 0)
        query_row = msa_int[0]
        keep = [0]  # always keep the query
        for i in range(1, n_seqs):
            # Identity = fraction of non-gap positions that match
            both_valid = (query_row >= 0) & (msa_int[i] >= 0)
            if both_valid.sum() == 0:
                continue
            identity = np.mean(query_row[both_valid] == msa_int[i][both_valid])
            if identity <= max_identity:
                keep.append(i)

        msa_int = msa_int[keep]
        n_kept = len(keep)
        print(f"  {name}: {n_seqs} → {n_kept} seqs (≤{max_identity:.0%} identity to query)")

        # Accumulate counts across columns
        for col in range(aln_len):
            valid = msa_int[:, col]
            valid = valid[valid >= 0]
            if len(valid) < 2:
                continue
            counts = np.bincount(valid, minlength=20)
            single_counts += counts
            outer = np.outer(counts, counts).astype(float)
            np.fill_diagonal(outer, counts * (counts - 1))
            pair_counts += outer
            total_columns += 1

    print(f"\nTotal: {total_columns} columns with ≥2 valid residues")
    print(f"Total pairs counted: {pair_counts.sum():.0f}")

    # Normalize and compute log-odds
    q = pair_counts / pair_counts.sum()
    p = single_counts / single_counts.sum()
    log_odds = np.log2((q + 1e-10) / (np.outer(p, p) + 1e-10))
    return np.round(log_odds * 2).astype(int), q, p

matrix, q_obs, p_bg = build_substitution_matrix(all_msas, max_identity=0.62)
print(f"\nMatrix computed! Shape: {matrix.shape}, score range: [{matrix.min()}, {matrix.max()}]")

In [ ]:
# Visualize — already in Dayhoff order since AA = 'CSTPAGNDEQHRKMILVFYW'
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(matrix, cmap='bwr_r', vmin=-6, vmax=6, aspect='equal')

ax.set_xticks(range(20))
ax.set_yticks(range(20))
ax.set_xticklabels(list(AA), fontsize=11, fontfamily='monospace')
ax.set_yticklabels(list(AA), fontsize=11, fontfamily='monospace')

for i in range(20):
    for j in range(20):
        color = 'white' if abs(matrix[i, j]) > 3 else 'black'
        ax.text(j, i, str(matrix[i, j]), ha='center', va='center', fontsize=7, color=color)

for b in GROUP_BOUNDS:
    ax.axhline(b - 0.5, color='black', linewidth=1.5)
    ax.axvline(b - 0.5, color='black', linewidth=1.5)

starts = [0] + GROUP_BOUNDS
ends = GROUP_BOUNDS + [20]
for s, e in zip(starts, ends):
    ax.text((s + e) / 2 - 0.5, -0.75, group_labels[s], ha='center', fontsize=9, fontweight='bold')

plt.colorbar(im, ax=ax, label='Log-odds score', shrink=0.8)
plt.tight_layout()
plt.show()

### Let's compare to the real BLOSUM62

Our matrix won't match exactly — BLOSUM62 uses many more MSAs and a 62% clustering threshold — but the overall pattern should be similar.

In [ ]:
# BLOSUM62 in Dayhoff order (CSTPAGNDEQHRKMILVFYW)
blosum62 = np.array([
 #  C  S  T  P  A  G  N  D  E  Q  H  R  K  M  I  L  V  F  Y  W
 [  9,-1,-1,-3, 0,-3,-3,-3,-4,-3,-3,-3,-3,-1,-1,-1,-1,-2,-2,-2],  # C
 [ -1, 4, 1,-1, 1, 0, 1, 0, 0, 0,-1,-1, 0,-1,-2,-2,-2,-2,-2,-3],  # S
 [ -1, 1, 5,-1, 0,-2, 0,-1,-1,-1,-2,-1,-1,-1,-1,-1, 0,-2,-2,-2],  # T
 [ -3,-1,-1, 7,-1,-2,-2,-1,-1,-1,-2,-2,-1,-2,-3,-3,-2,-4,-3,-4],  # P
 [  0, 1, 0,-1, 4, 0,-2,-2,-1,-1,-2,-1,-1,-1,-1,-1, 0,-2,-2,-3],  # A
 [ -3, 0,-2,-2, 0, 6, 0,-1,-2,-2,-2,-2,-2,-3,-4,-4,-3,-3,-3,-2],  # G
 [ -3, 1, 0,-2,-2, 0, 6, 1, 0, 0, 1, 0, 0,-2,-3,-3,-3,-3,-2,-4],  # N
 [ -3, 0,-1,-1,-2,-1, 1, 6, 2, 0,-1,-2,-1,-3,-3,-4,-3,-3,-3,-4],  # D
 [ -4, 0,-1,-1,-1,-2, 0, 2, 5, 2, 0, 0, 1,-2,-3,-3,-2,-3,-2,-3],  # E
 [ -3, 0,-1,-1,-1,-2, 0, 0, 2, 5, 0, 1, 1, 0,-3,-2,-2,-3,-1,-2],  # Q
 [ -3,-1,-2,-2,-2,-2, 1,-1, 0, 0, 8, 0,-1,-2,-3,-3,-3,-1, 2,-2],  # H
 [ -3,-1,-1,-2,-1,-2, 0,-2, 0, 1, 0, 5, 2,-1,-3,-2,-3,-3,-2,-3],  # R
 [ -3, 0,-1,-1,-1,-2, 0,-1, 1, 1,-1, 2, 5,-1,-3,-2,-2,-3,-2,-3],  # K
 [ -1,-1,-1,-2,-1,-3,-2,-3,-2, 0,-2,-1,-1, 5, 1, 2, 1, 0,-1,-1],  # M
 [ -1,-2,-1,-3,-1,-4,-3,-3,-3,-3,-3,-3,-3, 1, 4, 2, 3, 0,-1,-3],  # I
 [ -1,-2,-1,-3,-1,-4,-3,-4,-3,-2,-3,-2,-2, 2, 2, 4, 1, 0,-1,-2],  # L
 [ -1,-2, 0,-2, 0,-3,-3,-3,-2,-2,-3,-3,-2, 1, 3, 1, 4,-1,-1,-3],  # V
 [ -2,-2,-2,-4,-2,-3,-3,-3,-3,-3,-1,-3,-3, 0, 0, 0,-1, 6, 3, 1],  # F
 [ -2,-2,-2,-3,-2,-3,-2,-3,-2,-1, 2,-2,-2,-1,-1,-1,-1, 3, 7, 2],  # Y
 [ -2,-3,-2,-4,-3,-2,-4,-4,-3,-2,-2,-3,-3,-1,-3,-2,-3, 1, 2,11],  # W
], dtype=int)

# Verify symmetry
assert np.array_equal(blosum62, blosum62.T), "BLOSUM62 should be symmetric!"
print(f"BLOSUM62 loaded — {AA}")

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, mat, title in [(axes[0], matrix, 'Our Matrix (from MSA)'),
                       (axes[1], blosum62, 'BLOSUM62')]:
    im = ax.imshow(mat, cmap='bwr_r', vmin=-6, vmax=6, aspect='equal')
    ax.set_xticks(range(20))
    ax.set_yticks(range(20))
    ax.set_xticklabels(list(AA), fontsize=9, fontfamily='monospace')
    ax.set_yticklabels(list(AA), fontsize=9, fontfamily='monospace')
    ax.set_title(title, fontsize=14)
    for b in GROUP_BOUNDS:
        ax.axhline(b - 0.5, color='black', linewidth=1)
        ax.axvline(b - 0.5, color='black', linewidth=1)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Substitution Matrix Comparison (Dayhoff ordering)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: our scores vs BLOSUM62 (upper triangle)
ii, jj = np.triu_indices(20)
our_vals = matrix[ii, jj]
bl_vals = blosum62[ii, jj]

same = ii == jj
within = ~same & np.array([DAYHOFF_GROUPS[AA[i]] == DAYHOFF_GROUPS[AA[j]] for i, j in zip(ii, jj)])
between = ~same & ~within

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(bl_vals[between], our_vals[between], c='#9E9E9E', s=40, alpha=0.5, label='Between groups')
ax.scatter(bl_vals[within],  our_vals[within],  c='#4CAF50', s=50, alpha=0.7, label='Within Dayhoff group')
ax.scatter(bl_vals[same],    our_vals[same],    c='#2196F3', s=60, alpha=0.8, label='Same AA (diagonal)')

lims = [min(bl_vals.min(), our_vals.min()) - 1, max(bl_vals.max(), our_vals.max()) + 1]
ax.plot(lims, lims, 'k--', alpha=0.3, linewidth=1)

for k in np.where(same)[0]:
    if abs(our_vals[k]) > 3 or abs(bl_vals[k]) > 5:
        ax.annotate(AA[ii[k]], (bl_vals[k], our_vals[k]), fontsize=9,
                    fontweight='bold', xytext=(5, 5), textcoords='offset points')

corr = np.corrcoef(our_vals, bl_vals)[0, 1]
ax.legend(fontsize=10, loc='upper left')
ax.set_xlabel('BLOSUM62 score', fontsize=13)
ax.set_ylabel('Our matrix score', fontsize=13)
ax.set_title(f'Our Matrix vs BLOSUM62 (r = {corr:.3f})', fontsize=14)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Pearson correlation: {corr:.3f}")
print(f"Built from a SINGLE MSA, yet captures the same biochemistry as BLOSUM62!")

### Key observations

- **Block structure is visible**: similar amino acids cluster together with positive scores
- **Hydrophobic block** (M, I, L, V): these substitute freely in protein cores
- **Aromatic block** (F, Y, W): ring structures are interchangeable
- **Cysteine** stands out: high self-score because cysteines form disulfide bonds
- **Cross-group scores are negative**: radical substitutions are penalized

---
## Part 3 — Smith-Waterman Local Alignment

We need to **find** the best local alignment, allowing gaps (insertions/deletions).

### The recurrence

$$H[i,j] = \max \begin{cases} 0 & \text{(start fresh)} \\ H[i{-}1,j{-}1] + s(a_i, b_j) & \text{(match/mismatch)} \\ H[i{-}1,j] - d & \text{(gap in seq 2)} \\ H[i,j{-}1] - d & \text{(gap in seq 1)} \end{cases}$$

In [ ]:
def smith_waterman(seq1, seq2, subst_matrix, gap_penalty=10, score_only=False):
    """
    Smith-Waterman local alignment.

    Set score_only=True for fast database searches (no traceback, O(n) memory).
    """
    m, n = len(seq1), len(seq2)

    # Precompute all pairwise substitution scores
    idx1 = [AA_INDEX.get(c, -1) for c in seq1]
    idx2 = [AA_INDEX.get(c, -1) for c in seq2]
    ext = np.full((21, 21), -4.0)   # index -1 maps to row/col 20 → penalty of -4
    ext[:20, :20] = subst_matrix
    scores = ext[idx1][:, idx2]      # shape (m, n)

    if score_only:
        prev = np.zeros(n + 1)
        best = 0.0
        for i in range(m):
            curr = np.zeros(n + 1)
            for j in range(n):
                curr[j+1] = max(0,
                    prev[j]   + scores[i, j],
                    prev[j+1] - gap_penalty,
                    curr[j]   - gap_penalty)
            best = max(best, curr.max())
            prev = curr
        return best

    # Full DP with traceback
    H = np.zeros((m + 1, n + 1))
    tb = np.zeros((m + 1, n + 1), dtype=int)  # 0=stop, 1=diag, 2=up, 3=left

    best_score, best_pos = 0, (0, 0)
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            diag = H[i-1, j-1] + scores[i-1, j-1]
            up   = H[i-1, j]   - gap_penalty
            left = H[i, j-1]   - gap_penalty
            H[i, j] = v = max(0, diag, up, left)

            if   v == 0:    tb[i, j] = 0
            elif v == diag: tb[i, j] = 1
            elif v == up:   tb[i, j] = 2
            else:           tb[i, j] = 3

            if v > best_score:
                best_score, best_pos = v, (i, j)

    # Traceback
    aln1, aln2, mid = [], [], []
    i, j = best_pos
    while i > 0 and j > 0 and H[i, j] > 0:
        if tb[i, j] == 1:
            a, b = seq1[i-1], seq2[j-1]
            aln1.append(a); aln2.append(b)
            mid.append('|' if a == b else ('+' if scores[i-1, j-1] > 0 else '.'))
            i -= 1; j -= 1
        elif tb[i, j] == 2:
            aln1.append(seq1[i-1]); aln2.append('-'); mid.append(' '); i -= 1
        elif tb[i, j] == 3:
            aln1.append('-'); aln2.append(seq2[j-1]); mid.append(' '); j -= 1
        else:
            break

    alignment = (''.join(reversed(aln1)), ''.join(reversed(aln2)), ''.join(reversed(mid)))
    return best_score, alignment, H

print("smith_waterman() defined")
print("  Full alignment:  smith_waterman(s1, s2, matrix)")
print("  Score only:      smith_waterman(s1, s2, matrix, score_only=True)")

### Let's align two real sequences

We'll use BLOSUM62 for scoring (it's more robust than our single-MSA matrix).

In [ ]:
# RecA (DNA repair ATPase) vs RadA (distant homolog)
query  = "AIDENKQKALAAALGQIEKQFGKGSIMRLGEDRSMDVETISTGSLSLDIALGAGGLPMGRIVEIYGPESSGKTTLTLQVIAAAQREGKTCAFIDAEHALDPIYARKLGVDIDNLLCSQPDTGEQALEICDALARSGAVDVIVVDSVAALTPKAEIE"
target = "MSDQFIEQAFNALPGIEKSYGKGSSLRLADEGRSTGETISISGRLALETALQAGGLRGRISLYGTESSSGKTSVALQAIAAAQKTTGTCAFIDAEHALDPTYARQLGVIDIDDKLSKRPDKAGEQALEIFSYALERGDIDLIVVDSVAALTPKAEIE"

print(f"Query  ({len(query)} aa):  {query[:60]}...")
print(f"Target ({len(target)} aa): {target[:60]}...")
print()

t0 = time.time()
score, alignment, H = smith_waterman(query, target, blosum62)
elapsed = time.time() - t0
aln1, aln2, mid = alignment

print(f"Score: {score},  Time: {elapsed:.3f}s")
print()
print("Alignment:")
for start in range(0, len(aln1), 60):
    print(f"  Query:  {aln1[start:start+60]}")
    print(f"          {mid[start:start+60]}")
    print(f"  Target: {aln2[start:start+60]}")
    print()

In [ ]:
# Visualize the DP matrix
fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(H[1:, 1:], cmap='YlOrRd', aspect='auto')

step_x = max(1, len(target) // 40)
step_y = max(1, len(query) // 40)
ax.set_xticks(range(0, len(target), step_x))
ax.set_xticklabels([target[i] for i in range(0, len(target), step_x)], fontsize=8, fontfamily='monospace')
ax.set_yticks(range(0, len(query), step_y))
ax.set_yticklabels([query[i] for i in range(0, len(query), step_y)], fontsize=8, fontfamily='monospace')

ax.set_xlabel('Target sequence', fontsize=12)
ax.set_ylabel('Query sequence', fontsize=12)
ax.set_title(f'Smith-Waterman DP Matrix (best score = {score})', fontsize=14)
plt.colorbar(im, ax=ax, label='Alignment score', shrink=0.8)
plt.tight_layout()
plt.show()

### What does the score mean?

$$\text{Score} = \sum_{\text{aligned pairs}} s(a_i, b_j) - \sum_{\text{gaps}} d$$

The total score is the **total log-likelihood ratio** — how much more likely is this alignment under "homologous" vs "random"?

But — **is this score significant?** We need a p-value.

---
## Part 4 — P-values Revisited: Now With Alignment Scores

> "If I shuffle the target and re-run Smith-Waterman, what scores do I get by chance?"

This null distribution accounts for the substitution matrix, gap handling, and the "best local match" effect.

In [ ]:
n_shuffles = 500
target_letters = list(target)
null_scores = np.zeros(n_shuffles)

print(f"Running {n_shuffles} shuffled alignments...")
t0 = time.time()
for i in range(n_shuffles):
    shuffled = target_letters.copy()
    np.random.shuffle(shuffled)
    null_scores[i] = smith_waterman(query, ''.join(shuffled), blosum62, score_only=True)
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{n_shuffles} done ({time.time()-t0:.1f}s)")

print(f"Done in {time.time()-t0:.1f}s")
p_value_sw = np.mean(null_scores >= score)
print(f"\nObserved score: {score}")
print(f"Null mean ± std: {null_scores.mean():.1f} ± {null_scores.std():.1f}")
print(f"Empirical p-value: {p_value_sw}")

In [ ]:
# Fit a Gumbel (Extreme Value) distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(null_scores, bins=40, density=True, alpha=0.7, color='steelblue',
        edgecolor='white', label='Shuffled scores')

loc, scale = gumbel_r.fit(null_scores)
x = np.linspace(null_scores.min() - 5, max(null_scores.max(), score) + 10, 200)
ax.plot(x, gumbel_r.pdf(x, loc=loc, scale=scale), 'k-', linewidth=2,
        label=f'Gumbel fit (μ={loc:.1f}, β={scale:.1f})')

p_evd = 1 - gumbel_r.cdf(score, loc=loc, scale=scale)
ax.text(0.95, 0.95, f'EVD p-value: {p_evd:.2e}', transform=ax.transAxes,
        fontsize=12, ha='right', va='top', bbox=dict(boxstyle='round', facecolor='wheat'))

ax.axvline(score, color='red', linewidth=2, linestyle='--', label=f'Observed: {score}')
ax.set_xlabel('Smith-Waterman Score', fontsize=13)
ax.set_ylabel('Density', fontsize=13)
ax.set_title('Null Distribution of SW Scores\n(Karlin-Altschul: scores follow an Extreme Value Distribution)', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### Why an Extreme Value Distribution?

When you take the **maximum** of many random variables (which is what SW does — it finds the best local match), the result follows a **Gumbel distribution**. This is a fundamental result from extreme value theory.

**Karlin & Altschul (1990)** proved that for local alignment scores:

$$P(S \geq x) \approx 1 - e^{-Kmn \cdot e^{-\lambda x}}$$

BLAST uses this to compute E-values analytically without shuffling.

---
## Part 5 — Searching a Database: E-values and Multiple Testing

If you search against 4,300 E. coli proteins with a p-value threshold of 0.001, you'd still expect **4.3 false positives**. We need **E-values**.

In [ ]:
ECOLI_URL = "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=organism_id:83333+AND+reviewed:true"

print("Downloading E. coli K-12 proteome from UniProt...")
req = urllib.request.Request(ECOLI_URL, headers={'User-Agent': 'Python/tutorial'})
response = urllib.request.urlopen(req, timeout=30)
fasta_text = response.read().decode('utf-8')
print(f"Downloaded {len(fasta_text)} bytes")

In [ ]:
def parse_fasta(text):
    """Parse FASTA format into list of (name, sequence) tuples."""
    sequences, header, current = [], "", []
    for line in text.strip().split('\n'):
        if line.startswith('>'):
            if current:
                sequences.append((header, ''.join(current)))
            header = line[1:].split()[0]
            current = []
        else:
            current.append(line.strip())
    if current:
        sequences.append((header, ''.join(current)))
    return sequences

ecoli_proteins = parse_fasta(fasta_text)
print(f"Loaded {len(ecoli_proteins)} E. coli proteins")
lengths = [len(s) for _, s in ecoli_proteins]
print(f"Length range: {min(lengths)}-{max(lengths)} aa (median {np.median(lengths):.0f})")

### Experiment 1: Search with a random query (expect no real hits)

In [ ]:
random_query = random_protein(100)
subset = [(n, s) for n, s in ecoli_proteins if len(s) <= 512]
n_targets = len(subset)

print(f"Random query ({len(random_query)} aa): {random_query[:60]}...")
print(f"\nSearching against {n_targets} E. coli proteins...")

random_scores = []
t0 = time.time()
for i, (name, seq) in enumerate(subset):
    s = smith_waterman(random_query, seq[:200], blosum62, score_only=True)
    random_scores.append((s, name))
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{n_targets} ({time.time()-t0:.1f}s)")

random_scores.sort(reverse=True)
print(f"Done in {time.time()-t0:.1f}s")

print(f"\nTop 5 hits (random query — all noise):")
for s, name in random_scores[:5]:
    print(f"  {name}: score = {s:.0f}")

### Experiment 2: Search with a real query (RecA — expect to find homologs)

In [ ]:
reca_query = "AIDENKQKALAAALGQIEKQFGKGSIMRLGEDRSMDVETISTGSLSLDIALGAGGLPMGRIVEIYGPESSGKTTLTLQVIAAAQREGKTCAFIDAEHALDPIYARKLGVDIDNLLCSQPDTGEQALEICDALARSGAVDVIVVDSVAALTPKAEIE"
print(f"RecA query ({len(reca_query)} aa): {reca_query[:60]}...")

print(f"\nSearching against {n_targets} E. coli proteins...")
real_scores = []
t0 = time.time()
for i, (name, seq) in enumerate(subset):
    s = smith_waterman(reca_query, seq[:200], blosum62, score_only=True)
    real_scores.append((s, name))
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{n_targets} ({time.time()-t0:.1f}s)")

real_scores.sort(reverse=True)
print(f"Done in {time.time()-t0:.1f}s")

print(f"\nTop 10 hits (RecA query):")
for s, name in real_scores[:10]:
    print(f"  {name}: score = {s}")

In [ ]:
# Compare score distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, scores_list, title, color in [
    (axes[0], [s for s, _ in random_scores], 'Random Query', 'steelblue'),
    (axes[1], [s for s, _ in real_scores], 'RecA Query (has homologs)', 'darkgreen')
]:
    arr = np.array(scores_list)
    ax.hist(arr, bins=30, density=True, alpha=0.7, color=color, edgecolor='white')

    bg = arr[arr <= np.percentile(arr, 90)]
    loc, scale = gumbel_r.fit(bg)
    x = np.linspace(arr.min() - 5, arr.max() + 10, 200)
    ax.plot(x, gumbel_r.pdf(x, loc=loc, scale=scale), 'k-', linewidth=2, label='EVD fit (background)')
    ax.set_xlabel('Smith-Waterman Score', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=10)

plt.suptitle('Database Search: Score Distributions', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# E-values for RecA search
print("E-value computation for RecA search")
print("=" * 55)
print()

all_scores = np.array([s for s, _ in real_scores])
bg = all_scores[all_scores <= np.percentile(all_scores, 90)]
loc, scale = gumbel_r.fit(bg)
D = n_targets

print(f"Database size: {D}")
print(f"EVD parameters: μ = {loc:.2f}, β = {scale:.2f}")
print()
print(f"{'Rank':<6} {'Protein':<20} {'Score':<8} {'P-value':<12} {'E-value':<12} {'Significant?'}")
print("-" * 75)

for rank, (s, name) in enumerate(real_scores[:15], 1):
    p = 1 - gumbel_r.cdf(s, loc=loc, scale=scale)
    e = p * D
    sig = "***" if e < 0.001 else "**" if e < 0.01 else "*" if e < 1 else ""
    print(f"{rank:<6} {name:<20} {s:<8.0f} {p:<12.2e} {e:<12.4f} {sig}")

print()
print("Significance: *** E < 0.001,  ** E < 0.01,  * E < 1")

### Understanding E-values

| E-value | Meaning |
|---------|---------|
| 0.001 | Expect this by chance once in 1,000 database searches → **very confident** |
| 0.01 | Once in 100 searches → **confident** |
| 1 | Expect 1 false positive per search → **borderline** |
| 10 | Expect 10 hits this good by chance → **not significant** |

$$E = p \times D$$

BLAST uses a more sophisticated version that accounts for database size and sequence lengths in the EVD parameters themselves, but the core idea is identical.

---
## Summary: The Full Pipeline

We've built, from scratch, the core statistical framework behind tools like BLAST:

1. **Percent identity** → too naive, treats all substitutions equally
2. **Substitution matrix** → log-odds scores from evolutionary data (BLOSUM)
3. **Smith-Waterman** → finds optimal local alignment using the matrix
4. **P-value** → "is this score surprising?" via null distribution
5. **E-value** → corrects for multiple comparisons ($E = p \times D$)

### What BLAST adds:
- **Heuristic seeding** — find short exact matches first, extend only promising ones (~1000× faster)
- **Pre-computed EVD parameters** — avoids expensive shuffling
- **Gapped extension** — banded DP for speed

The statistical foundations are exactly what we built here.

### Key equations:

**Log-odds score:** $s_{ab} = \log_2 \frac{q_{ab}}{p_a \cdot p_b}$

**Alignment score:** $S = \sum s(a_i, b_j) - \text{gap penalties}$

**E-value:** $E = p \times D$

---
*Tutorial built for demonstration purposes. For production bioinformatics, use established tools like BLAST, HMMER, MMseqs2, etc.*